In [4]:
#%pip install ultralytics
#%pip install torch==2.0.0
#%pip install numpy==1.26.4
#%pip install torchvision==0.15.2
#%pip install scikit-learn

### Split Data

In [5]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Define paths

def create_standart_ultranalytics_dataset(old_directory, new_directory, ):

    labels = [ f.path.split("/")[-1] for f in os.scandir(old_directory + '/images') if f.is_dir() ]

    if not os.path.isdir(new_directory):
        os.makedirs(new_directory, exist_ok=True)

    for label in labels:

        image_dir = old_directory + '/images/' + label
        label_dir = old_directory + '/labels/' + label

        new_image_dir = new_directory + '/images/'
        new_label_dir = new_directory + '/labels/'

        train_dir = [new_image_dir + '/train/' + label, new_label_dir + '/train/' + label]
        val_dir = [new_image_dir + '/val/' + label, new_label_dir + '/val/' + label]
        test_dir = [new_image_dir + '/test/' + label, new_label_dir + '/test/' + label]

        # Create directories
        for directory in [x for xs in [train_dir, val_dir, test_dir] for x in xs] :
            os.makedirs(directory, exist_ok=True)
        # Get all image files

        images = [f for f in os.listdir(image_dir) if f.endswith('.JPG')]
        # Split the dataset

        train_images, temp_images = train_test_split(images, test_size=0.2, random_state=42)
        val_images, test_images = train_test_split(temp_images, test_size=0.5, random_state=42)
        
        # Move images to respective directories

        for img in train_images:
            shutil.copy(os.path.join(image_dir, img), train_dir[0])
            shutil.copy(os.path.join(label_dir, img.replace('.JPG', '.txt')), train_dir[1])
        for img in val_images:
            shutil.copy(os.path.join(image_dir, img), val_dir[0])
            shutil.copy(os.path.join(label_dir, img.replace('.JPG', '.txt')), val_dir[1])
        for img in test_images:
            shutil.copy(os.path.join(image_dir, img), test_dir[0])
            shutil.copy(os.path.join(label_dir, img.replace('.JPG', '.txt')), test_dir[1])

In [6]:
create_standart_ultranalytics_dataset(
    old_directory="../Datasets/DatasetObjectDetectionOnly",
    new_directory="Dataset",
)

#### TREINO DE YOLO NO DATASET DE 3Classes

In [7]:
import time
import os
from ultralytics import YOLO


epochs = 200
batch_size = 16
config_path = './config.yaml'

save_dir = "./Resultados"

##### YOLO 12

In [8]:
#load model
model = YOLO("yolo12n.pt")

In [9]:
start = time.time()

#use model
model.train(data=config_path, epochs=epochs, batch=batch_size, project = save_dir) #Train

end = time.time()
print("Tempo: " + str(end - start))

Ultralytics 8.3.168 🚀 Python-3.8.20 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=Tr

train: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreinoYoloCNN/YOLO/Dataset/labels/train/Disease... 5529 images, 0 backgrounds, 0 corrupt: 100%|██████████| 5529/5529 [00:07<00:00, 740.18it/s]


train: New cache created: /home/vmarmelo/Documents/Workspace/ModelosTreinoYoloCNN/YOLO/Dataset/labels/train/Disease.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 277.9±111.3 MB/s, size: 18.6 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreinoYoloCNN/YOLO/Dataset/labels/val/Disease... 691 images, 0 backgrounds, 0 corrupt: 100%|██████████| 691/691 [00:00<00:00, 794.17it/s]

val: New cache created: /home/vmarmelo/Documents/Workspace/ModelosTreinoYoloCNN/YOLO/Dataset/labels/val/Disease.cache


...

    200/200      4.82G      1.498     0.6937      1.312        131        640: 100%|██████████| 346/346 [00:51<00:00,  6.75it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [00:03<00:00,  7.01it/s]

                   all        691       8403      0.771      0.778      0.826      0.429



200 epochs completed in 3.101 hours.
Optimizer stripped from Resultados/train/weights/last.pt, 5.5MB
Optimizer stripped from Resultados/train/weights/best.pt, 5.5MB

Validating Resultados/train/weights/best.pt...
Ultralytics 8.3.168 🚀 Python-3.8.20 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
YOLOv12n summary (fused): 159 layers, 2,557,313 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 22/22 [00:07<00:00,  2.96it/s]


                   all        691       8403      0.763      0.787      0.833      0.436
             Black_rot        691       8403      0.763      0.787      0.833      0.436
Speed: 0.4ms preprocess, 1.3ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to Resultados/train
Tempo: 11197.452974557877


In [ ]:
model.save("./YOLO12n_OnlyDetection.pt")

In [14]:
import pandas as pd

results_df = pd.DataFrame([], columns=["Precision", "Recall", "F1-Score", "mAP50", "mAP50-95", "Loss", "Fitness", "Inference Speed (ms)"])

for conf_value in [0.3]: 

    results = model.val(data="config.yaml", conf=conf_value, batch=128, plots=True, name="./Validation")

    f1score = 2 * (results.results_dict['metrics/precision(B)'] * results.results_dict['metrics/recall(B)']) / (results.results_dict['metrics/precision(B)'] + results.results_dict['metrics/recall(B)'])
    
    results_df.loc["Model with conf of " + str(conf_value)] = [
        results.results_dict['metrics/precision(B)'],
        results.results_dict['metrics/recall(B)'],
        f1score,
        results.results_dict['metrics/mAP50(B)'],
        results.results_dict['metrics/mAP50-95(B)'],
        results.speed['loss'],
        results.results_dict['fitness'],
        results.speed['inference'],
    ]

Ultralytics 8.3.168 🚀 Python-3.8.20 torch-2.0.1+cu117 CUDA:0 (Tesla V100-DGXS-32GB, 32508MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 425.1±92.1 MB/s, size: 18.6 KB)


val: Scanning /home/vmarmelo/Documents/Workspace/ModelosTreinoYoloCNN/YOLO/Dataset/labels/val/Disease.cache... 691 images, 0 backgrounds, 0 corrupt: 100%|██████████| 691/691 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:09<00:00,  1.61s/it]


                   all        691       8403      0.755      0.795      0.804      0.451
             Black_rot        691       8403      0.755      0.795      0.804      0.451
Speed: 2.0ms preprocess, 4.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to Resultados/Validation


In [15]:
results_df

,Precision,Recall,F1-Score,mAP50,mAP50-95,Loss,Fitness,Inference Speed (ms)
Model with conf of 0.3,0.755314,0.794954,0.774627,0.803519,0.450939,0.000194,0.486197,4.205936


In [1]:
results = model.val(data="config.yaml", conf=conf_value, batch=128, plots=True, name="./Validation")

NameError: name 'model' is not defined